<a href="https://colab.research.google.com/github/vibeshvidya/MedicalChatbot/blob/main/MediBot_RAG_Assignment_24_Jun.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
GOOGLE_API_KEY=""
PINECONE_API_KEY=""

In [ ]:
!pip install langchain==0.3.26
!pip install sentence-transformers==4.1.0
!pip install pypdf==5.6.1
!pip install python-dotenv==1.1.0
!pip install langchain-pinecone==0.2.8
!pip install "langchain-google-genai<3"
!pip install langchain-community==0.3.26

In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()

    return documents

In [6]:
extracted_docs = load_pdf_files('/content/drive/MyDrive/IIT_Palakkad_Advanced_AI')

In [ ]:
extracted_docs

In [8]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs):

    minimal_docs = []

    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

In [9]:
minimal_docs = filter_to_minimal_docs(extracted_docs)

In [ ]:
minimal_docs

In [11]:
#CHUNKING

def text_split(minimal_docs):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    texts_chunk = text_splitter.split_documents(minimal_docs)

    return texts_chunk

In [12]:
texts_chunk = text_split(minimal_docs)

In [13]:
len(texts_chunk)

5859

In [ ]:
texts_chunk[1232]

In [ ]:
texts_chunk[1233]

In [16]:
#EMBEDDING

from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():

    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )

    return embeddings


In [ ]:
embedding = download_embeddings()

In [18]:
vector = embedding.embed_query("My name is Vibesh")
len(vector)

384

In [19]:
import os

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [20]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [21]:
from pinecone import ServerlessSpec

index_name = "medibot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

index = pc.Index(index_name)

In [22]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [23]:
#RETRIEVER

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)



In [24]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [25]:
retrieved_docs = retriever.invoke("What is Acne?")

In [ ]:
retrieved_docs

In [35]:

from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [ ]:

response = chatModel.invoke("Hello")
response
#print(response.content)


In [29]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [30]:
system_prompt=(
    """
    You are a Medical Assistant for question-answer tasks.
    Use the following pieces of retrieved context to answer the question.
    If you don't know the answer say that you don't know
    Use three sentences maximum and keep the answer concise.
    \n\n
    {context}
    """
)

In [31]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]
)

In [32]:
question_answer_chain = create_stuff_documents_chain(chatModel,prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)


In [ ]:
#response = rag_chain.invoke({"input":"What is acne"})
response = rag_chain.invoke({"input":"What are the remedies of Acne"})

In [ ]:
#response
response['answer']